In [3]:
!nvidia-smi --query-gpu=name --format=csv

from google.colab import drive
drive.mount('/content/drive')

import os
SAVE = '/content/drive/MyDrive/SongNet'
print(os.listdir(SAVE))

name
Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['split.npz', 'tracks_used.csv']


In [4]:
%cd /content
!wget -q --show-progress https://os.unil.cloud.switch.ch/fma/fma_metadata.zip
!wget -q --show-progress https://os.unil.cloud.switch.ch/fma/fma_small.zip
!unzip -q fma_metadata.zip
!unzip -q fma_small.zip
!ls fma_small/*/*.mp3 | wc -l

/content
fma_metadata.zip    100%[===================>] 341.81M  27.2MB/s    in 5.5s    
fma_small.zip       100%[===================>]   7.15G  56.7MB/s    in 74s     
8000


In [5]:
import os, time, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import librosa, librosa.display
import torch, torch.nn as nn, torch.nn.functional as F
from joblib import Parallel, delayed
from sklearn.metrics import classification_report, confusion_matrix

SAVE = '/content/drive/MyDrive/SongNet'
SR, N_FFT, HOP, N_MELS, FRAMES = 22050, 2048, 512, 128, 1280     # same settings as VS Code
device = 'cuda'
torch.manual_seed(42); np.random.seed(42)

# 1) load YOUR cleaned song list and YOUR split from VS Code
df = pd.read_csv(f'{SAVE}/tracks_used.csv', index_col=0)
df['path'] = [f'/content/fma_small/{t // 1000:03d}/{t:06d}.mp3' for t in df.index]   # Colab location
GENRES = sorted(df.genre.unique())
y = df.label.values
s = np.load(f'{SAVE}/split.npz')
train_idx, val_idx, test_idx = s['train_idx'], s['val_idx'], s['test_idx']
MEAN, STD = float(s['MEAN']), float(s['STD'])
print('songs:', len(df), '| train/val/test:', len(train_idx), len(val_idx), len(test_idx),
      f'| mean {MEAN:.1f}, std {STD:.1f}')

# 2) same preprocessing function as in VS Code
def song_to_mel(path, fixed_length=True):
    try:
        y_, _ = librosa.load(path, sr=SR, mono=True, duration=30)
        mel = librosa.power_to_db(librosa.feature.melspectrogram(
            y=y_, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS), ref=np.max)
    except Exception:
        return None
    if not fixed_length:
        return mel
    if mel.shape[1] < FRAMES // 2:
        return None
    if mel.shape[1] < FRAMES:
        mel = np.pad(mel, ((0, 0), (0, FRAMES - mel.shape[1])), constant_values=mel.min())
    return mel[:, :FRAMES].astype(np.float16)

# 3) rebuild the spectrograms, same order as VS Code
start = time.time()
mels = Parallel(n_jobs=-1, verbose=5)(delayed(song_to_mel)(p) for p in df.path)
assert all(m is not None for m in mels), 'a song failed that worked in VS Code!'
X = np.stack(mels); del mels
print('X shape:', X.shape, f'| took {(time.time() - start) / 60:.1f} min')

songs: 7994 | train/val/test: 5595 1599 800 | mean -43.9, std 16.3


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  14 tasks      | elapsed:   31.4s
[Parallel(n_jobs=-1)]: Done  68 tasks      | elapsed:   38.0s
[Parallel(n_jobs=-1)]: Done 158 tasks      | elapsed:   45.7s
[Parallel(n_jobs=-1)]: Done 284 tasks      | elapsed:   59.3s
[Parallel(n_jobs=-1)]: Done 446 tasks      | elapsed:  1.3min
[Parallel(n_jobs=-1)]: Done 644 tasks      | elapsed:  1.6min
[Parallel(n_jobs=-1)]: Done 878 tasks      | elapsed:  2.0min
[Parallel(n_jobs=-1)]: Done 1148 tasks      | elapsed:  2.5min
[Parallel(n_jobs=-1)]: Done 1454 tasks      | elapsed:  3.0min
[Parallel(n_jobs=-1)]: Done 1796 tasks      | elapsed:  3.6min
[Parallel(n_jobs=-1)]: Done 2174 tasks      | elapsed:  4.3min
[Parallel(n_jobs=-1)]: Done 2588 tasks      | elapsed:  5.0min
[Parallel(n_jobs=-1)]: Done 3038 tasks      | elapsed:  5.7min
[Parallel(n_jobs=-1)]: Done 3524 tasks      | elapsed:  6.6min
[Parallel(n_jobs=-1)]: Done 4046 tasks      | ela

X shape: (7994, 128, 1280) | took 14.6 min


In [6]:
class SongNet(nn.Module):
    def __init__(self, n_genres=8):
        super().__init__()
        def block(c_in, c_out):
            return nn.Sequential(nn.Conv1d(c_in, c_out, kernel_size=3, padding=1),
                                 nn.BatchNorm1d(c_out), nn.ReLU(),
                                 nn.MaxPool1d(2), nn.Dropout(0.3))
        self.cnn = nn.Sequential(block(N_MELS, 256), block(256, 256), block(256, 256))
        self.gru = nn.GRU(256, 128, num_layers=2, batch_first=True, dropout=0.3)
        self.out = nn.Linear(128, n_genres)

    def forward(self, x):
        h = self.cnn(x)
        h, _ = self.gru(h.transpose(1, 2))
        return self.out(h)

def clip_probs(step_scores):
    return F.softmax(step_scores, dim=-1).mean(dim=1)

def loss_fn(step_scores, labels):
    probs = clip_probs(step_scores)
    return F.nll_loss(torch.log(probs + 1e-8), labels), probs

class MelDataset(torch.utils.data.Dataset):
    def __init__(self, indices, crop=None):
        self.indices, self.crop = indices, crop
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        j = self.indices[i]
        mel = (X[j].astype(np.float32) - MEAN) / STD
        if self.crop:
            start = np.random.randint(0, FRAMES - self.crop + 1)
            mel = mel[:, start:start + self.crop]
        return torch.from_numpy(mel), int(y[j])

train_loader = torch.utils.data.DataLoader(MelDataset(train_idx, crop=640), batch_size=32, shuffle=True, num_workers=2)
val_loader   = torch.utils.data.DataLoader(MelDataset(val_idx),  batch_size=32, num_workers=2)
test_loader  = torch.utils.data.DataLoader(MelDataset(test_idx), batch_size=32, num_workers=2)

model = SongNet(len(GENRES)).to(device)
print('parameters:', sum(p.numel() for p in model.parameters()))

# timing test on the GPU
xb, yb = next(iter(train_loader))
t = time.time(); loss, _ = loss_fn(model(xb.to(device)), yb.to(device)); loss.backward(); torch.cuda.synchronize()
print(f'one batch: {time.time() - t:.2f} s | loss before training: {loss.item():.3f} (random = 2.079)')


parameters: 742152
one batch: 3.72 s | loss before training: 2.062 (random = 2.079)


In [7]:
def run_epoch(loader, train):
    model.train(train)                           # train mode (dropout on) or eval mode (off)
    total_loss, correct, count = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            loss, probs = loss_fn(model(xb), yb)
            if train:                            # learn: backpropagation + update weights
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * len(yb)
            correct += (probs.argmax(1) == yb).sum().item()
            count += len(yb)
    return total_loss / count, 100 * correct / count

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=3)

history, best_acc, waited = [], 0, 0
for epoch in range(1, 81):
    t = time.time()
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader, train=False)
    scheduler.step(va_loss)                      # lower the learning rate if validation stalls
    history.append((tr_loss, tr_acc, va_loss, va_acc))
    note = ''
    if va_acc > best_acc:                        # best so far -> save it to Drive
        best_acc, waited, note = va_acc, 0, '  <- best, saved'
        torch.save({'model': model.state_dict(), 'mean': MEAN, 'std': STD, 'genres': GENRES},
                   f'{SAVE}/songnet.pt')
    else:
        waited += 1
    print(f'epoch {epoch:2d} | train {tr_acc:5.1f}% | val {va_acc:5.1f}% | '
          f'val loss {va_loss:.3f} | {time.time() - t:.0f}s{note}')
    if waited == 15:                             # early stopping
        print('no improvement for 15 epochs -> stopping'); break

pd.DataFrame(history, columns=['train_loss', 'train_acc', 'val_loss', 'val_acc']).to_csv(f'{SAVE}/history.csv', index=False)
print(f'\nbest validation accuracy: {best_acc:.1f}%  (model saved to Drive: SongNet/songnet.pt)')

epoch  1 | train  34.7% | val  39.3% | val loss 1.600 | 9s  <- best, saved
epoch  2 | train  39.7% | val  41.1% | val loss 1.588 | 11s  <- best, saved
epoch  3 | train  42.6% | val  44.6% | val loss 1.521 | 11s  <- best, saved
epoch  4 | train  45.3% | val  42.2% | val loss 1.606 | 10s
epoch  5 | train  47.5% | val  42.7% | val loss 1.562 | 11s
epoch  6 | train  49.8% | val  47.3% | val loss 1.441 | 12s  <- best, saved
epoch  7 | train  50.5% | val  47.5% | val loss 1.426 | 9s  <- best, saved
epoch  8 | train  52.0% | val  51.8% | val loss 1.361 | 11s  <- best, saved
epoch  9 | train  53.2% | val  50.5% | val loss 1.428 | 11s
epoch 10 | train  53.8% | val  52.4% | val loss 1.378 | 9s  <- best, saved
epoch 11 | train  55.0% | val  52.3% | val loss 1.339 | 11s
epoch 12 | train  55.6% | val  52.3% | val loss 1.354 | 11s
epoch 13 | train  57.4% | val  52.2% | val loss 1.319 | 9s
epoch 14 | train  56.8% | val  54.2% | val loss 1.336 | 11s  <- best, saved
epoch 15 | train  58.8% | val  53.4%